# Boat detector training on Colab

Runs `train.py` from this repository on a Colab GPU. Runs are written straight to Google Drive, so a disconnected session resumes from the latest `last.pt` by rerunning all cells.

1. **Runtime > Change runtime type**: pick a GPU.
2. Upload a dataset bundle zip (containing `<dataset>/dataset.yaml`, `images/`, `labels/`) to Drive.
3. Edit the parameters below and run all cells.

Presets: see `EXPERIMENTS` in `train.py`, e.g. `v4s_frozen` for a frozen backbone at 640 px. Training uses color recordings only (see README); bundle a color dataset.

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/boat_color_bundle.zip'  # dataset bundle on Drive
DATASET = 'yolo_dataset_v4_color'                  # dataset folder inside the zip
EXPERIMENT = 'v5_s'                                # train.py preset
NAME = 'boat_color_v5_s'                           # run name
OVERRIDES = ['batch=32']                           # extra Ultralytics args
RUNS = '/content/drive/MyDrive/boat_runs'          # persistent run directory

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/work /content/repo
!git clone -q --depth 1 https://github.com/arminekberi/shipDetection-distancePrediction.git /content/repo
!pip install -q ultralytics==8.4.138
!mkdir -p /content/work && unzip -q "$ZIP_PATH" -d /content/work

In [ ]:
# Point dataset.yaml at the extracted copy.
import pathlib, yaml
data = pathlib.Path('/content/work') / DATASET / 'dataset.yaml'
settings = yaml.safe_load(data.read_text())
settings['path'] = str(data.parent)
data.write_text(yaml.safe_dump(settings, sort_keys=False))
print(data.read_text())

In [ ]:
overrides = ' '.join(f'-o {o}' for o in OVERRIDES)
!cd /content/repo && python train.py {EXPERIMENT} --name {NAME} --data {data} --device 0 --project {RUNS} --resume {overrides}

Checkpoints: `RUNS/NAME/weights/best.pt`. Keep `args.yaml` and `results.csv` with the weights for provenance, and evaluate on independently reviewed recordings before promoting a checkpoint.